In [ ]:
import numpy as np
import pandas as pd
import optuna
from collections import defaultdict
import optunahub
from sklearn.metrics import v_measure_score, adjusted_mutual_info_score, adjusted_rand_score

from src.utilities.reranking import re_ranking
from src.clustering_algorithms.chinese_whispers import chinese_whispers
from src.utilities.image2identity import *

In [ ]:
split_dir = "openset_database_split"
ds_name = "Dataset you want to run optuna on"

sim_query_db = np.load(f"{split_dir}/{ds_name}_sim_query_db.npy", allow_pickle=True)
sim_query_query = np.load(f"{split_dir}/{ds_name}_sim_query_query.npy", allow_pickle=True)
y_true_query = np.load(f"{split_dir}/{ds_name}_y_true_query.npy", allow_pickle=True)
y_true_db = np.load(f"{split_dir}/{ds_name}_y_true_db.npy", allow_pickle=True)
known_mask_query = np.load(f"{split_dir}/{ds_name}_known_mask_query.npy", allow_pickle=True)
unknown_mask_query = np.load(f"{split_dir}/{ds_name}_unknown_mask_query.npy", allow_pickle=True)

print("sim_query_db   :", sim_query_db.shape)
print("sim_query_query:", sim_query_query.shape)
print("y_true_query   :", y_true_query.shape)
print("y_true_db      :", y_true_db.shape)
print("known queries  :", known_mask_query.sum())
print("unknown queries:", unknown_mask_query.sum())

In [ ]:
def relabel_sequential(labels):
    """
    Convert arbitrary cluster strings / ids into 0..K-1 for ARI.
    """
    unique = {}
    out = np.empty(len(labels), dtype=np.int32)
    nxt = 0
    for i, x in enumerate(labels):
        if x not in unique:
            unique[x] = nxt
            nxt += 1
        out[i] = unique[x]
    return out

In [ ]:
RR_SPECIES = {"LynxID2025", "SalamanderID2025", "SeaTurtleID2022"}

def run_fullgraph_cw_pipeline_on_split(
    sim_query_db,
    sim_query_query,
    y_true_db,
    top_r,
    tau_attach,
    tau_margin,
    cw_threshold,
    cw_top_k,
    cw_iterations,
    cw_k_core_degree,
    ds_name="LynxID2025",
    use_rerank=False,
    rr_k1=20,
    rr_k2=6,
    rr_lambda=0.3,
):
    identity_names, identity_scores = compute_identity_scores(
        sim_test_train=sim_query_db,
        train_ids=y_true_db,
        top_r=top_r
    )

    best_identity_names, best_scores, second_best_scores, margins = summarize_best_identities(
        identity_names, identity_scores
    )

    if use_rerank and ds_name in RR_SPECIES:
        sim_for_cw = re_ranking(
            sim_query_query,
            k1=rr_k1,
            k2=rr_k2,
            lambda_value=rr_lambda,
        )
    else:
        sim_for_cw = sim_query_query

    cluster_ids = chinese_whispers(
        sim_for_cw,
        threshold=cw_threshold,
        iterations=cw_iterations,
        top_k=cw_top_k,
        k_core_degree=cw_k_core_degree,
        seed=42,
    )

    cluster_votes = defaultdict(lambda: defaultdict(float))

    for i in range(len(cluster_ids)):
        c = int(cluster_ids[i])

        if (best_scores[i] >= tau_attach) and (margins[i] >= tau_margin):
            full_id = str(best_identity_names[i])
            short_id = full_id.split("_")[-1]
            cluster_votes[c][short_id] += float(best_scores[i])

    cluster_to_label = {}
    for c, votes in cluster_votes.items():
        if votes:
            best_label = max(votes.items(), key=lambda kv: kv[1])[0]
            cluster_to_label[c] = f"cluster_{ds_name}_{best_label}"

    y_pred = np.empty(len(cluster_ids), dtype=object)
    for i in range(len(cluster_ids)):
        c = int(cluster_ids[i])
        y_pred[i] = cluster_to_label.get(
            c,
            f"cluster_{ds_name}_new_{c}"
        )

    return y_pred, cluster_ids, best_scores, margins

In [ ]:
def objective_fullgraph_cw(trial):
    top_r = trial.suggest_int("top_r", 1, 5)
    tau_attach = trial.suggest_float("tau_attach", 0.1, 0.9, step=0.01)
    tau_margin = trial.suggest_float("tau_margin", 0.1, 0.9, step=0.01)
    cw_threshold = trial.suggest_float("cw_threshold", 0.12, 0.68, step=0.01)
    cw_top_k = trial.suggest_int("cw_top_k", 7, 60)
    cw_iterations = trial.suggest_int("cw_iterations", 5, 30)
    cw_k_core_degree = trial.suggest_int("cw_k_core_degree", 1, 5)
    use_rerank = True

    if use_rerank and ds_name in RR_SPECIES:
        rr_k1 = trial.suggest_int("rr_k1", 5, 50)
        rr_k2 = trial.suggest_int("rr_k2", 1, 15)
        rr_lambda = trial.suggest_float("rr_lambda", 0.1, 0.9)
    else:
        rr_k1, rr_k2, rr_lambda = 20, 6, 0.3

    y_pred, cluster_ids, best_scores, margins = run_fullgraph_cw_pipeline_on_split(
        sim_query_db=sim_query_db,
        sim_query_query=sim_query_query,
        y_true_db=y_true_db,
        top_r=top_r,
        tau_attach=tau_attach,
        tau_margin=tau_margin,
        cw_threshold=cw_threshold,
        cw_top_k=cw_top_k,
        cw_iterations=cw_iterations,
        cw_k_core_degree=cw_k_core_degree,
        ds_name=ds_name,
        use_rerank=use_rerank,
        rr_k1=rr_k1,
        rr_k2=rr_k2,
        rr_lambda=rr_lambda,
    )

    ari = adjusted_rand_score(
        relabel_sequential(y_true_query),
        relabel_sequential(y_pred)
    )

    v_measure = v_measure_score(
        relabel_sequential(y_true_query),
        relabel_sequential(y_pred)
    )

    trial.set_user_attr("num_clusters", int(len(np.unique(cluster_ids))))
    trial.set_user_attr("mean_best_score", float(best_scores.mean()))
    trial.set_user_attr("mean_margin", float(margins.mean()))
    trial.set_user_attr("used_rerank", bool(use_rerank))
    
    return 0.6 * ari + 0.4 * v_measure 

In [ ]:
# module = optunahub.load_module(package="samplers/auto_sampler")
# sampler = module.AutoSampler(seed=42)

study_fullgraph_cw = optuna.create_study(
    direction="maximize",
    study_name=f"fullgraph_cw_{ds_name}",
    sampler=optuna.samplers.TPESampler(seed=42, multivariate=True),
)

study_fullgraph_cw.optimize(
    objective_fullgraph_cw,
    n_trials=2000
)

In [ ]:
print("Best Objective Score:", study_fullgraph_cw.best_value)
print("Best params:")
for k, v in study_fullgraph_cw.best_params.items():
    print(f"  {k}: {v}")

print("\nBest trial user attrs:")
for k, v in study_fullgraph_cw.best_trial.user_attrs.items():
    print(f"  {k}: {v}")

best_fullgraph = study_fullgraph_cw.best_params

if best_fullgraph.get("use_rerank", False) and ds_name in RR_SPECIES:
    rr_k1 = best_fullgraph["rr_k1"]
    rr_k2 = best_fullgraph["rr_k2"]
    rr_lambda = best_fullgraph["rr_lambda"]
else:
    rr_k1, rr_k2, rr_lambda = 20, 6, 0.3

y_pred_best_fullgraph, cluster_ids_best_fullgraph, best_scores_best_fullgraph, margins_best_fullgraph = run_fullgraph_cw_pipeline_on_split(
    sim_query_db=sim_query_db,
    sim_query_query=sim_query_query,
    y_true_db=y_true_db,
    top_r=best_fullgraph["top_r"],
    tau_attach=best_fullgraph["tau_attach"],
    tau_margin=best_fullgraph["tau_margin"],
    cw_threshold=best_fullgraph["cw_threshold"],
    cw_top_k=best_fullgraph["cw_top_k"],
    cw_iterations=best_fullgraph["cw_iterations"],
    cw_k_core_degree=best_fullgraph["cw_k_core_degree"],
    ds_name=ds_name,
    use_rerank=best_fullgraph.get("use_rerank", False),
    rr_k1=rr_k1,
    rr_k2=rr_k2,
    rr_lambda=rr_lambda,
)


best_ari_fullgraph = adjusted_rand_score(
    relabel_sequential(y_true_query),
    relabel_sequential(y_pred_best_fullgraph)
)


print("Recomputed best ARI:", best_ari_fullgraph)
print("Num predicted clusters:", len(np.unique(cluster_ids_best_fullgraph)))
print("Mean best score:", best_scores_best_fullgraph.mean())
print("Mean margin:", margins_best_fullgraph.mean())